In [ ]:
#######FIX PATH######
main_path = "파일경로"
######################

In [2]:
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import networkx as nx
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
import numpy as np
import os 
import matplotlib.pyplot as plt
from collections import OrderedDict
from torcheval.metrics.functional import binary_f1_score   
from torcheval.metrics.functional import binary_auprc
import shap
import shutil
from time import time,localtime, strftime
import argparse
import matplotlib.pyplot as plt

In [3]:
exp = f'{main_path}/data/final_exp_count_dat_15685_177.txt'
phenotype = f'{main_path}/data/final_pheno_dat_177.txt'
exp_df = pd.read_table(exp,index_col=0)
phenotype = pd.read_table(phenotype)
phenotype.columns = ['Name','Class','batch_effect']
phenotype = phenotype[['Name','Class']]
exp_df = exp_df.T.reset_index().rename(columns={'index':'Name'})
exp_df = pd.merge(exp_df,phenotype,on='Name')
exp_df.set_index('Name',inplace=True)

In [4]:
for i in range(1,6):
    with open(f'{main_path}/pathway/new_pathway_'+str(i)+'.pickle', 'rb') as f:
        globals()['adj{}'.format(i)]=pickle.load(f)

In [5]:
exp_data = exp_df.drop('Class',axis=1).T
PP = exp_df[exp_df.Class=='PP'].index.tolist()
NN = exp_df[exp_df.Class=='NN'].index.tolist()
PN = exp_df[exp_df.Class=='PN'].index.tolist()

In [ ]:
# lab_data : ID	State	Tissue

In [6]:
exp_df = exp_df[exp_df.Class.isin(['PP','PN'])]

In [7]:
lab_data = exp_df[['Class']].reset_index()
lab_data.columns = ['ID','State']

In [8]:
PP_data=exp_data.loc[:,PP]
Con_data=exp_data.loc[:,PN]
# Con_data=exp_data.loc[:,NN]

mask_GeneToH2=adj5.T
mask_H2ToH3=adj4.T
mask_H3ToH4=adj3.T
mask_H4ToH5=adj2.T
mask_H5ToH6=adj1.T

exp_data=pd.concat([PP_data,Con_data],axis=1)

lab_data["label"]=0

list_temp=[]
for j,i in enumerate(lab_data["State"]):
    if "PP" in i:
        lab_data["label"][j]=1
    elif "PN" in i:
        lab_data["label"][j]=0
    else:
        lab_data["label"][j]=0
lab_data.index=lab_data["ID"]
# gene intersect만 남기기

genes=list(mask_GeneToH2.index)

set_genes=list(set(list(exp_data.index))&set(genes))

input_data=exp_data.loc[set_genes]
print("gene length"+str(len(set_genes)))

mask_GeneToH2=mask_GeneToH2.loc[set_genes]

lab_data=lab_data.loc[list(input_data.columns)]

input_data

gene length8148



A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

Name,GSM2882402,GSM2882404,GSM2882406,GSM2882408,GSM2211694,GSM2211697,GSM2211703,GSM2211708,GSM2211714,GSM1930383,...,GSM3427984,GSM3427986,GSM3427988,GSM3427990,GSM3427992,GSM3427994,GSM3427996,GSM3427999,GSM3428001,GSM3428003
NDUFA1,10.264746,10.090446,10.552179,10.045564,10.901201,10.895153,11.276302,11.082326,11.205864,9.605487,...,9.671709,9.511417,9.475151,9.086571,9.231483,9.849427,9.223390,9.232651,9.314263,9.576078
EDA2R,7.518178,7.312351,3.771244,7.044286,5.832159,5.928545,6.124195,5.207586,5.892435,6.811111,...,4.675448,4.528980,3.549754,5.116968,4.987427,4.801265,4.129804,3.497881,2.682150,5.356948
MUC16,1.792481,4.284502,0.000000,2.230422,1.773859,3.300538,2.387267,2.225857,1.773859,2.352098,...,6.321524,6.129111,6.986923,5.741795,0.232096,4.486275,7.271255,5.930743,8.466188,5.861473
SLC36A1,9.192140,9.060528,8.822918,8.891793,9.025896,9.559860,8.800926,9.293199,8.926008,8.473063,...,7.516544,7.422607,8.053528,7.109990,7.640699,7.444364,7.912292,7.709984,7.934656,7.475429
GREM2,0.764160,0.000000,6.842887,0.000000,6.799842,6.758930,8.295649,6.419613,7.117791,6.920970,...,6.405860,6.523880,5.259155,6.636705,6.894446,5.806727,6.596068,5.678417,5.848675,6.965140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GPKOW,8.845011,8.615168,8.912582,8.894622,9.071050,9.165461,9.452211,9.491693,9.537317,8.160003,...,8.533541,8.917209,8.516230,8.595480,8.766666,8.690597,8.667282,8.560886,8.234585,8.705374
DSC1,10.049208,10.595684,14.761840,8.963336,13.320746,13.870001,14.476541,14.579481,14.209914,14.726891,...,15.986372,15.647763,15.228393,15.413104,15.744954,15.556736,15.744954,15.867849,15.867849,15.744954
SIRT4,3.917552,3.965668,3.555893,4.087967,4.451814,5.233294,4.789087,4.332406,5.387151,2.973402,...,4.590620,4.273293,3.549754,4.319830,3.724599,4.486275,3.682107,3.265042,4.136124,4.159654
ABR,10.624399,10.562620,11.824790,10.801152,11.988233,12.101964,11.481310,12.058904,12.199930,10.821060,...,10.496096,10.497581,10.775612,10.947579,10.761424,10.594562,10.982429,10.830955,10.605543,10.827157


In [9]:
outpath = f'{main_path}/psoriasis/'
os.makedirs(outpath+'/input/',exist_ok=True)
os.makedirs(outpath+'/output/',exist_ok=True)
os.makedirs(outpath+'/result/',exist_ok=True)

In [10]:

with open(f"{outpath}/input/input_data.pickle","wb") as f:
    pickle.dump(input_data,f,pickle.HIGHEST_PROTOCOL)


with open(f"{outpath}/input/lab_data.pickle","wb") as f:
    pickle.dump(lab_data,f,pickle.HIGHEST_PROTOCOL)

#저장 / 불러오기 

masks = [mask_GeneToH2, mask_H2ToH3, mask_H3ToH4, mask_H4ToH5, mask_H5ToH6]

with open(f"{outpath}/input/masks.pickle","wb") as f:
    pickle.dump(masks,f,pickle.HIGHEST_PROTOCOL)



print(mask_GeneToH2.shape)
print(mask_H2ToH3.shape)
print(mask_H3ToH4.shape)
print(mask_H4ToH5.shape)
print(mask_H5ToH6.shape)

masks = [mask_GeneToH2, mask_H2ToH3, mask_H3ToH4, mask_H4ToH5, mask_H5ToH6]

# dataset_size = input_data.shape[1]
# train_size = int(dataset_size * 0.85)
# test_size = int(dataset_size * 0.15)

# test_size=test_size+(dataset_size-train_size-test_size)
dataset_size = input_data.shape[1]
train_size = int(dataset_size * 0.8)
valid_size = int(dataset_size * 0.1)
test_size = int(dataset_size * 0.1)

train_size=train_size+(dataset_size-train_size-test_size-valid_size)

t_exp=input_data.T

#train_dataset, test_dataset = random_split(input_data.columns, [train_size, test_size])

train_dataset, valid_dataset, test_dataset = random_split(input_data.columns, [train_size, valid_size, test_size])
x_dataTensor = torch.from_numpy(t_exp.iloc[train_dataset.indices].to_numpy()).float()
y_dataTensor = torch.from_numpy(lab_data[["label"]].iloc[train_dataset.indices].to_numpy()).float()

print("x_dataTensor.shape : ", x_dataTensor.shape)
print("y_dataTensor.shape : ", y_dataTensor.shape)

x_validTensor = torch.from_numpy(t_exp.iloc[valid_dataset.indices].to_numpy()).float()
y_validTensor = torch.from_numpy(lab_data[["label"]].iloc[valid_dataset.indices].to_numpy()).float()
print("x_validTensor.shape : ", x_validTensor.shape)
print("y_validTensor.shape : ", y_validTensor.shape)

x_testTensor = torch.from_numpy(t_exp.iloc[test_dataset.indices].to_numpy()).float()
y_testTensor = torch.from_numpy(lab_data[["label"]].iloc[test_dataset.indices].to_numpy()).float()
print("x_testTensor.shape : ", x_testTensor.shape)
print("y_testTensor.shape : ", y_testTensor.shape)

print((y_dataTensor.count_nonzero().item()/y_dataTensor.size()[0])*100)
print((y_validTensor.count_nonzero().item()/y_validTensor.size()[0])*100)
print((y_testTensor.count_nonzero().item()/y_testTensor.size()[0])*100)




(8148, 1610)
(1610, 1116)
(1116, 502)
(502, 167)
(167, 29)
x_dataTensor.shape :  torch.Size([97, 8148])
y_dataTensor.shape :  torch.Size([97, 1])
x_validTensor.shape :  torch.Size([11, 8148])
y_validTensor.shape :  torch.Size([11, 1])
x_testTensor.shape :  torch.Size([11, 8148])
y_testTensor.shape :  torch.Size([11, 1])
57.73195876288659
63.63636363636363
54.54545454545454


In [11]:
train_dataset, valid_dataset, test_dataset = random_split(input_data.columns, [train_size, valid_size, test_size])
x_dataTensor = torch.from_numpy(t_exp.iloc[train_dataset.indices].to_numpy()).float()
y_dataTensor = torch.from_numpy(lab_data[["label"]].iloc[train_dataset.indices].to_numpy()).float()

print("x_dataTensor.shape : ", x_dataTensor.shape)
print("y_dataTensor.shape : ", y_dataTensor.shape)

x_validTensor = torch.from_numpy(t_exp.iloc[valid_dataset.indices].to_numpy()).float()
y_validTensor = torch.from_numpy(lab_data[["label"]].iloc[valid_dataset.indices].to_numpy()).float()
print("x_validTensor.shape : ", x_validTensor.shape)
print("y_validTensor.shape : ", y_validTensor.shape)

x_testTensor = torch.from_numpy(t_exp.iloc[test_dataset.indices].to_numpy()).float()
y_testTensor = torch.from_numpy(lab_data[["label"]].iloc[test_dataset.indices].to_numpy()).float()
print("x_testTensor.shape : ", x_testTensor.shape)
print("y_testTensor.shape : ", y_testTensor.shape)

print((y_dataTensor.count_nonzero().item()/y_dataTensor.size()[0])*100)
print((y_validTensor.count_nonzero().item()/y_validTensor.size()[0])*100)
print((y_testTensor.count_nonzero().item()/y_testTensor.size()[0])*100)


x_dataTensor.shape :  torch.Size([97, 8148])
y_dataTensor.shape :  torch.Size([97, 1])
x_validTensor.shape :  torch.Size([11, 8148])
y_validTensor.shape :  torch.Size([11, 1])
x_testTensor.shape :  torch.Size([11, 8148])
y_testTensor.shape :  torch.Size([11, 1])
58.76288659793815
36.36363636363637
72.72727272727273


In [12]:
from sklearn.model_selection import StratifiedKFold
str_kf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
split=[]
for train_index, test_index in str_kf.split(x_dataTensor,y_dataTensor):
    X_train, X_valid = x_dataTensor[train_index], x_dataTensor[test_index]
    y_train, y_valid = y_dataTensor[train_index], y_dataTensor[test_index]
    split.append([X_train,y_train,X_valid,y_valid])
with open(f"{outpath}/input/split.pickle","wb") as f:
    pickle.dump(split,f,pickle.HIGHEST_PROTOCOL)

In [13]:

import pickle
with open(f"{outpath}/input/train_X_Tensor.pickle","wb") as f:
    pickle.dump(x_dataTensor,f,pickle.HIGHEST_PROTOCOL)
with open(f"{outpath}/input/valid_X_Tensor.pickle","wb") as f:
    pickle.dump(x_validTensor,f,pickle.HIGHEST_PROTOCOL)
with open(f"{outpath}/input/test_X_Tensor.pickle","wb") as f:
    pickle.dump(x_testTensor,f,pickle.HIGHEST_PROTOCOL)

with open(f"{outpath}/input/train_y_Tensor.pickle","wb") as f:
    pickle.dump(y_dataTensor,f,pickle.HIGHEST_PROTOCOL)
with open(f"{outpath}/input/valid_y_Tensor.pickle","wb") as f:
    pickle.dump(y_validTensor,f,pickle.HIGHEST_PROTOCOL)
with open(f"{outpath}/input/test_y_Tensor.pickle","wb") as f:
    pickle.dump(y_testTensor,f,pickle.HIGHEST_PROTOCOL)
